# Ti Adatom Descent on a 5×5 TiN Island (MACE)

> **D. G. Sangiovanni, A. B. Mei, D. Edström, L. Hultman, V. Chirita, I. Petrov, and J. E. Greene**,
> "Effects of surface vibrations on interlayer mass transport: Ab initio molecular dynamics investigation of Ti
> adatom descent pathways and rates from TiN/TiN(001) islands", Physical Review B 97, 035406 (2018).
> [DOI: 10.1103/PhysRevB.97.035406](https://doi.org/10.1103/PhysRevB.97.035406)

Relaxes a Ti adatom at three sites of the paper's Fig. 6 on the cell created in the
[structure notebook](defect_surface_island_titanium_nitride.ipynb): **a**, the fourfold hollow on the island
next to its edge; **c**, atop the edge N atom; **i**, atop the terrace N atom in front of the edge, the site
that extends the island. The energies of c and i relative to a, and the adsorption energy at a, are compared
with the paper's. The paper computed them with DFT (GGA); here every relaxation uses the
[MACE-MP-0](https://github.com/ACEsuit/mace) machine-learned force field with the D3 dispersion correction.

The four relaxations are 458-atom MACE runs, meant to be executed natively; the notebook took 16 to 34 minutes on an Apple M1 Pro CPU in two runs.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_surface_island_titanium_nitride.ipynb), which saves the island cell and the three adatom cells to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.3 (or use the defaults).
1. Click "Run" > "Run All" to run all cells.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only), material names, force-field and relaxation parameters.
1. Load the materials by name from the `uploads` folder.
1. Build the MACE calculator.
1. Relax the island cell; measure the island contraction and the surface ripple.
1. Relax the adatom at a, c and i.
1. Compute the energy of an isolated Ti atom.
1. Compare with the paper: energies relative to a, the adsorption energy at a, and Fig. 6.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.calculators.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Material names

In [ ]:
# Names saved by defect_surface_island_titanium_nitride.ipynb.
ISLAND_MATERIAL_NAME = "TiN(001) 12x12x3 island 5x5"
ADATOM_MATERIAL_NAMES = {
    "a": "TiN(001) 12x12x3 island 5x5 + Ti a (FFH island)",
    "c": "TiN(001) 12x12x3 island 5x5 + Ti c (atop-N edge)",
    "i": "TiN(001) 12x12x3 island 5x5 + Ti i (atop-N terrace)",
}

### 1.3. Parameters

In [ ]:
FOLDER = "./uploads"

MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"
MACE_DISPERSION = True
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"

FMAX = 0.02  # eV/Å, every atom free
MAX_STEPS = 1000

ISOLATED_ATOM_BOX = 15.0  # Å, edge of the cubic box around the isolated Ti atom

## 2. Load the materials

In [ ]:
from mat3ra.notebooks_utils.material import load_materials_from_folder

materials_by_name = {material.name: material for material in load_materials_from_folder(FOLDER)}
slab_with_island = materials_by_name[ISLAND_MATERIAL_NAME]
materials_with_adatom = {site: materials_by_name[name] for site, name in ADATOM_MATERIAL_NAMES.items()}

for material in [slab_with_island, *materials_with_adatom.values()]:
    print(f"{material.name}: {material.basis.number_of_atoms} atoms")

## 3. Build the MACE calculator

In [ ]:
import importlib.util

from mat3ra.notebooks_utils.calculators.mlff import create_mlff_calculator

dispersion_available = importlib.util.find_spec("torch_dftd") is not None
dispersion_active = MACE_DISPERSION and dispersion_available
if MACE_DISPERSION and not dispersion_available:
    print("torch-dftd is not available here: MACE runs WITHOUT the D3 dispersion correction.")

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": dispersion_active,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)

## 4. Relax the island cell
### 4.1. Relax

Every atom is free, as in the paper's 0 K relaxations.

In [ ]:
import numpy as np
from mat3ra.made.tools.calculate import calculate_total_energy
from mat3ra.made.tools.convert import to_ase
from mat3ra.notebooks_utils.workflows.relaxation import relax_material

slab_with_island_relaxed = relax_material(slab_with_island, calculator, fmax=FMAX, max_steps=MAX_STEPS)
energy_slab_with_island = calculate_total_energy(slab_with_island_relaxed, calculator)
print(f"E(slab + island) = {energy_slab_with_island:.3f} eV")

### 4.2. Island contraction and surface ripple

The island's Ti–Ti spacing along its diagonals and Ti–N spacing along its medians, against the same spacings as
built on the bulk lattice; the ripple is the height of the N atoms above the Ti atoms in the exposed top plane of
the terrace (Fig. 2 of the paper).

In [ ]:
from mat3ra.made.tools.analyze.other import get_atom_indices_by_layer

built, relaxed = to_ase(slab_with_island), to_ase(slab_with_island_relaxed)
symbols = np.array(built.get_chemical_symbols())
planes = get_atom_indices_by_layer(slab_with_island)
island, terrace = np.array(planes[-1]), np.array(planes[-2])

island_x, island_y = built.positions[island, 0], built.positions[island, 1]
x_values = (island_x.min(), (island_x.min() + island_x.max()) / 2, island_x.max())
y_values = (island_y.min(), (island_y.min() + island_y.max()) / 2, island_y.max())
grid = {(column, row): island[np.argmin(np.hypot(island_x - x, island_y - y))]
        for column, x in enumerate(x_values) for row, y in enumerate(y_values)}
island_rows = {
    "Ti–Ti along diagonals": [(grid[0, 0], grid[2, 2]), (grid[0, 2], grid[2, 0])],
    "Ti–N along medians": [(grid[0, 1], grid[2, 1]), (grid[1, 0], grid[1, 2])],
}
spacings_per_row = np.sqrt(len(island)) - 1

island_contraction = {}
for label, pairs in island_rows.items():
    as_built, after = (np.mean([atoms.get_distance(*pair) for pair in pairs]) / spacings_per_row
                       for atoms in (built, relaxed))
    island_contraction[label] = 100 * (after / as_built - 1)
    print(f"{label}: {after:.3f} Å, {island_contraction[label]:+.1f} % against {as_built:.3f} Å as built")

island_xy = built.positions[island, :2]
exposed_terrace = [index for index in terrace
                   if not np.isclose(np.linalg.norm(island_xy - built.positions[index, :2], axis=1), 0).any()]
ripple = (np.mean([relaxed.positions[index, 2] for index in exposed_terrace if symbols[index] == "N"])
          - np.mean([relaxed.positions[index, 2] for index in exposed_terrace if symbols[index] == "Ti"]))
print(f"Ripple of the exposed terrace ({len(exposed_terrace)} atoms): {ripple:.3f} Å")

## 5. Relax the adatom at a, c and i

Each relaxation starts from the site placed in the structure notebook; the adatom's displacement from that site
shows whether it stayed there.

In [ ]:
energies = {}
for site, material in materials_with_adatom.items():
    material_relaxed = relax_material(material, calculator, fmax=FMAX, max_steps=MAX_STEPS)
    energies[site] = calculate_total_energy(material_relaxed, calculator)

    placed, final = to_ase(material), to_ase(material_relaxed)
    adatom = len(final) - 1
    displacement = np.linalg.norm(final.positions[adatom] - placed.positions[adatom])
    distances = np.sort(final.get_distances(adatom, range(adatom), mic=True))[:3]
    print(f"{site}: E = {energies[site]:.3f} eV, adatom moved {displacement:.2f} Å from its site, "
          f"three shortest distances {np.round(distances, 3)} Å")

## 6. Isolated Ti atom

The reference of the adsorption energy, Eq. (4) of the paper: one Ti atom in a cubic box, same calculator.

In [ ]:
from ase import Atoms

adatom_element = materials_with_adatom["a"].basis.elements.values[-1]
energy_isolated_adatom = calculate_total_energy(Atoms(adatom_element, cell=[ISOLATED_ATOM_BOX] * 3, pbc=True), calculator)
print(f"E({adatom_element}, isolated) = {energy_isolated_adatom:.3f} eV")

## 7. Compare with the paper
### 7.1. Energies relative to a, adsorption energy at a, island relaxation

The paper's energies of c and i are read off Fig. 6 (±0.02 and ±0.05 eV); the adsorption energy at a, the
isolated Ti atom energy it uses, the island contraction and the ripple are from Sec. III.A. The adsorption energy
is printed with this notebook's isolated-atom energy and with the paper's.

In [ ]:
# Sangiovanni et al. (2018): Fig. 6 upper panel (a-i, read off the axis); Sec. III.A
PAPER_FIG6 = {"a": 0.0, "b": 0.10, "c": -0.15, "d": -0.20, "e": -0.25, "f": -0.50, "g": -1.13, "h": -2.25, "i": -2.65}
PAPER_ADSORPTION_ENERGY = -2.81
PAPER_ISOLATED_ADATOM_ENERGY = -2.275
PAPER_ISLAND_CONTRACTION = {"Ti–Ti along diagonals": -6.8, "Ti–N along medians": -5.2}
PAPER_RIPPLE = 0.19

relative_energies = {site: energy - energies["a"] for site, energy in energies.items()}
print(f"{'site':<6}{'E - E_a (eV)':>14}{'paper':>8}{'deviation':>12}")
for site in ("c", "i"):
    deviation = 100 * (relative_energies[site] / PAPER_FIG6[site] - 1)
    print(f"{site:<6}{relative_energies[site]:>14.3f}{PAPER_FIG6[site]:>8.2f}{deviation:>+11.1f}%")

print()
for label, isolated_energy in (("this notebook's", energy_isolated_adatom), ("the paper's", PAPER_ISOLATED_ADATOM_ENERGY)):
    adsorption_energy = energies["a"] - energy_slab_with_island - isolated_energy
    deviation = 100 * (adsorption_energy / PAPER_ADSORPTION_ENERGY - 1)
    print(f"E_ads(a) with {label} E_Ti ({isolated_energy:.3f} eV): {adsorption_energy:.3f} eV, "
          f"paper {PAPER_ADSORPTION_ENERGY:.2f} eV, deviation {deviation:+.1f}%")

print()
for label, contraction in island_contraction.items():
    print(f"{label}: {contraction:+.1f} %, paper {PAPER_ISLAND_CONTRACTION[label]:+.1f} %")
print(f"Ripple: {ripple:.2f} Å, paper {PAPER_RIPPLE:.2f} Å")

### 7.2. Fig. 6

In [ ]:
import plotly.graph_objects as go

figure = go.Figure()
figure.add_trace(go.Scatter(x=list(PAPER_FIG6), y=list(PAPER_FIG6.values()), mode="lines+markers",
                            name="Sangiovanni et al. (2018)"))
figure.add_trace(go.Scatter(x=list(relative_energies), y=list(relative_energies.values()), mode="markers",
                            marker={"size": 12}, name="this notebook (MACE-MP-0)"))
figure.update_layout(title="Ti adatom descent from the 5×5 island edge (Fig. 6)", xaxis_title="state",
                     yaxis_title="energy relative to a (eV)")
figure.show()

## References

[1] D. G. Sangiovanni, A. B. Mei, D. Edström, L. Hultman, V. Chirita, I. Petrov, J. E. Greene, "Effects of surface
vibrations on interlayer mass transport: Ab initio molecular dynamics investigation of Ti adatom descent pathways
and rates from TiN/TiN(001) islands", Phys. Rev. B 97, 035406 (2018).
[DOI: 10.1103/PhysRevB.97.035406](https://doi.org/10.1103/PhysRevB.97.035406)

[2] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace